In [1]:
# ================================================================
# 🧬 GroupDNA — WhatsApp Group Personality & Activity Analyzer
# ================================================================
# Project: GroupDNA Minor Project
#
# Constraints:
#   ✅ NumPy allowed and used for the actual activity matrix
#   ❌ pandas
#   ❌ Matplotlib
#   ❌ Seaborn
#   ❌ re
#   ❌ Counter / defaultdict
#
# Features:
#   1. WhatsApp chat parser
#   2. Group overview and participant leaderboard
#   3. Media and deleted-message analysis
#   4. Message-length analysis
#   5. Word-frequency analysis
#   6. Most active day and hour
#   7. NumPy activity heatmap
#   8. Night-owl analysis
#   9. Response-time analysis
#  10. Longest silent streak
#  11. Personality archetypes
#  12. Attractive console dashboard
# ================================================================

import os
import numpy as np
from datetime import datetime, timedelta


# ================================================================
# 1. DISPLAY HELPERS
# ================================================================

WIDTH = 76

def line(char="─"):
    print(char * WIDTH)

def heading(title, emoji=""):
    print()
    line("═")
    print(f"  {emoji} {title}")
    line("═")

def section(title):
    print()
    print(f"  ✦ {title}")
    line("─")

def bar(value, maximum, width=30):
    """Create a text-based progress bar without plotting libraries."""
    if maximum <= 0:
        return "░" * width

    filled = int((value / maximum) * width)
    filled = max(0, min(width, filled))

    return "█" * filled + "░" * (width - filled)

def percentage(part, total):
    if total == 0:
        return 0.0
    return (part / total) * 100

def safe_divide(numerator, denominator):
    if denominator == 0:
        return 0.0
    return numerator / denominator


# ================================================================
# 2. FIND OR UPLOAD THE DATASET
# ================================================================

possible_paths = [
    "/content/hostel_bois(4).txt",
    "/content/hostel_bois.txt",
    "hostel_bois(4).txt",
    "hostel_bois.txt"
]

FILE_PATH = None

for path in possible_paths:
    if os.path.isfile(path):
        FILE_PATH = path
        break

# Ask the user to upload the dataset if it is not already available.
if FILE_PATH is None:
    try:
        from google.colab import files

        print("📂 Please upload hostel_bois.txt or hostel_bois(4).txt")
        uploaded = files.upload()

        for filename in uploaded:
            if filename.lower().endswith(".txt"):
                FILE_PATH = "/content/" + filename
                break

    except ImportError:
        pass

if FILE_PATH is None or not os.path.isfile(FILE_PATH):
    raise FileNotFoundError(
        "Dataset not found. Place hostel_bois.txt in the notebook "
        "working directory and run this cell again."
    )

print(f"\n📁 Dataset loaded: {os.path.basename(FILE_PATH)}")


# ================================================================
# 3. INITIALIZE PARTICIPANTS AND ANALYTICS STORAGE
# ================================================================

EXPECTED_PARTICIPANTS = [
    "Rahul",
    "Priya",
    "Aman",
    "Karan",
    "Neha",
    "Vikas"
]

participants = {}

for name in EXPECTED_PARTICIPANTS:
    participants[name] = {
        "messages": 0,
        "text_messages": 0,
        "media": 0,
        "deleted": 0,
        "total_characters": 0,
        "uppercase_messages": 0,
        "exclamation_messages": 0,
        "drama_messages": 0,
        "questions": 0,
        "active_days": set(),
        "hour_counts": [0] * 24,
        "day_counts": [0] * 7,
        "night_messages": 0,
        "words": {},
        "message_lengths": [],
        "timestamps": []
    }

all_messages = []

system_messages = 0
empty_lines = 0
malformed_lines = 0
continuation_lines = 0
raw_line_count = 0


# ================================================================
# 4. DATE AND MESSAGE PARSING HELPERS
# ================================================================

def looks_like_date(line_text):
    """
    Detect the DD/MM/YY prefix without using regular expressions.
    Example: 12/04/24, 23:14 - Rahul: Hello
    """

    if len(line_text) < 8:
        return False

    first_eight = line_text[:8]

    if (
        first_eight[2] != "/"
        or first_eight[5] != "/"
    ):
        return False

    return (
        first_eight[0:2].isdigit()
        and first_eight[3:5].isdigit()
        and first_eight[6:8].isdigit()
    )


def parse_timestamp(timestamp_text):
    """Convert a WhatsApp timestamp into a datetime object."""

    formats = [
        "%d/%m/%y, %H:%M",
        "%d/%m/%y, %I:%M %p",
        "%d/%m/%y, %I:%M:%S %p"
    ]

    for fmt in formats:
        try:
            return datetime.strptime(timestamp_text.strip(), fmt)
        except ValueError:
            continue

    return None


def split_chat_line(line_text):
    """
    Return (timestamp, sender, message).

    If the line is a system message or cannot be parsed,
    return None.
    """

    if " - " not in line_text:
        return None

    timestamp_text, remainder = line_text.split(" - ", 1)

    timestamp = parse_timestamp(timestamp_text)

    if timestamp is None:
        return None

    # A valid participant line must contain a sender/message separator.
    # Split only once so colons inside message text are preserved.
    if ": " not in remainder:
        return {
            "timestamp": timestamp,
            "sender": None,
            "message": remainder.strip()
        }

    sender, message = remainder.split(": ", 1)

    sender = sender.strip()
    message = message.strip()

    if not sender:
        return None

    return {
        "timestamp": timestamp,
        "sender": sender,
        "message": message
    }


# ================================================================
# 5. READ THE FILE AND HANDLE MULTILINE MESSAGES
# ================================================================

with open(FILE_PATH, "r", encoding="utf-8-sig") as file:
    raw_lines = file.readlines()

raw_line_count = len(raw_lines)

parsed_records = []
current_record = None

for raw_line in raw_lines:
    text = raw_line.rstrip("\r\n")

    if not text.strip():
        empty_lines += 1
        continue

    # A timestamped line starts a new record.
    if looks_like_date(text):

        if current_record is not None:
            parsed_records.append(current_record)

        current_record = {
            "raw": text,
            "continuation_count": 0
        }

    else:
        # Non-date lines continue the previous message.
        if current_record is not None:
            current_record["raw"] += "\n" + text
            current_record["continuation_count"] += 1
            continuation_lines += 1
        else:
            malformed_lines += 1

if current_record is not None:
    parsed_records.append(current_record)


# ================================================================
# 6. CLASSIFY MESSAGES
# ================================================================

for record in parsed_records:

    parsed = split_chat_line(record["raw"])

    if parsed is None:
        malformed_lines += 1
        continue

    timestamp = parsed["timestamp"]
    sender = parsed["sender"]
    message = parsed["message"]

    # System messages have a timestamp but no valid sender field.
    if sender is None:
        system_messages += 1
        continue

    # Preserve participant names exactly as they appear in the file.
    # This also allows unexpected senders to be reported.
    if sender not in participants:
        participants[sender] = {
            "messages": 0,
            "text_messages": 0,
            "media": 0,
            "deleted": 0,
            "total_characters": 0,
            "uppercase_messages": 0,
            "exclamation_messages": 0,
            "drama_messages": 0,
            "questions": 0,
            "active_days": set(),
            "hour_counts": [0] * 24,
            "day_counts": [0] * 7,
            "night_messages": 0,
            "words": {},
            "message_lengths": [],
            "timestamps": []
        }

    person = participants[sender]

    person["messages"] += 1
    person["timestamps"].append(timestamp)
    person["active_days"].add(timestamp.date())
    person["hour_counts"][timestamp.hour] += 1
    person["day_counts"][timestamp.weekday()] += 1

    # Store each participant message for chronological analysis.
    all_messages.append({
        "timestamp": timestamp,
        "sender": sender,
        "message": message
    })

    normalized_message = message.strip()
    lowered_message = normalized_message.lower()

    is_media = "<media omitted>" in lowered_message
    is_deleted = lowered_message == "this message was deleted"

    if is_media:
        person["media"] += 1
        continue

    if is_deleted:
        person["deleted"] += 1
        continue

    # Ordinary text messages only from this point onward.
    person["text_messages"] += 1
    person["total_characters"] += len(normalized_message)
    person["message_lengths"].append(len(normalized_message))

    # Night activity: 11 PM through 3:59 AM.
    if timestamp.hour >= 23 or timestamp.hour < 4:
        person["night_messages"] += 1

    # Uppercase detection: at least one alphabetic character
    # and all alphabetic characters are uppercase.
    alphabetic_characters = []

    for character in normalized_message:
        if character.isalpha():
            alphabetic_characters.append(character)

    is_uppercase = (
        len(alphabetic_characters) > 0
        and all(character.isupper() for character in alphabetic_characters)
    )

    if is_uppercase:
        person["uppercase_messages"] += 1

    # Exclamation-message detection.
    exclamation_count = normalized_message.count("!")

    if exclamation_count >= 2:
        person["exclamation_messages"] += 1

    # FIXED DRAMA RULE:
    # Count a message once if it is uppercase OR contains >= 2 ! marks.
    if is_uppercase or exclamation_count >= 2:
        person["drama_messages"] += 1

    if "?" in normalized_message:
        person["questions"] += 1


# Sort all messages chronologically before response-time calculations.
all_messages.sort(key=lambda item: item["timestamp"])


# ================================================================
# 7. WORD FREQUENCY
# ================================================================

# Common words that are not very informative for this analysis.
# "guys" is intentionally NOT included so it can appear in results.
STOP_WORDS = {
    "the", "a", "an", "is", "am", "are", "was", "were",
    "be", "been", "being", "to", "of", "and", "or", "in",
    "on", "at", "for", "from", "with", "by", "as", "it",
    "this", "that", "these", "those", "i", "me", "my", "we",
    "our", "you", "your", "he", "she", "they", "them",
    "his", "her", "their", "what", "which", "who", "when",
    "where", "why", "how", "can", "could", "would", "should",
    "will", "just", "not", "no", "yes", "do", "does", "did",
    "have", "has", "had", "if", "then", "than", "so", "but",
    "because", "about", "into", "out", "up", "down", "there",
    "here", "all", "any", "some", "more", "very", "too",
    "also", "get", "got", "go", "going", "went", "come",
    "coming", "now", "ok", "okay", "yeah", "ya", "yep",
    "hmm", "hm", "oh", "lol", "haha", "hahaha"
}


def tokenize(message):
    """
    Split a message into words using character checks.
    No regular expressions or NLP packages are used.
    """

    words = []
    current_word = ""

    for character in message.lower():

        if character.isalpha() or character.isdigit():
            current_word += character

        else:
            if current_word:
                words.append(current_word)
                current_word = ""

    if current_word:
        words.append(current_word)

    return words


for item in all_messages:

    sender = item["sender"]
    message = item["message"]

    lowered = message.strip().lower()

    if (
        "<media omitted>" in lowered
        or lowered == "this message was deleted"
    ):
        continue

    person = participants[sender]

    for word in tokenize(message):

        if len(word) < 2:
            continue

        if word in STOP_WORDS:
            continue

        current_count = person["words"].get(word, 0)
        person["words"][word] = current_count + 1


# ================================================================
# 8. RESPONSE-GAP ANALYSIS
# ================================================================

# A response gap is the time between consecutive messages
# from different senders.
#
# This is an approximation of response time, not proof that
# the second person was directly replying to the first.

response_gaps = {}
all_response_gaps = []

for name in participants:
    response_gaps[name] = []

for index in range(1, len(all_messages)):

    previous = all_messages[index - 1]
    current = all_messages[index]

    if previous["sender"] == current["sender"]:
        continue

    gap = current["timestamp"] - previous["timestamp"]

    if gap.total_seconds() < 0:
        continue

    gap_minutes = gap.total_seconds() / 60

    response_gaps[current["sender"]].append(gap_minutes)
    all_response_gaps.append(gap_minutes)


# ================================================================
# 9. LONGEST SILENT STREAK
# ================================================================

if all_messages:
    first_date = min(item["timestamp"].date() for item in all_messages)
    last_date = max(item["timestamp"].date() for item in all_messages)
else:
    first_date = datetime(2024, 4, 1).date()
    last_date = datetime(2024, 5, 30).date()

total_days = (last_date - first_date).days + 1

for name in participants:

    active_dates = participants[name]["active_days"]

    longest_streak = 0
    current_streak = 0
    streak_start = None
    longest_start = None
    longest_end = None

    day = first_date

    while day <= last_date:

        if day not in active_dates:

            if current_streak == 0:
                streak_start = day

            current_streak += 1

            if current_streak > longest_streak:
                longest_streak = current_streak
                longest_start = streak_start
                longest_end = day

        else:
            current_streak = 0
            streak_start = None

        day += timedelta(days=1)

    participants[name]["longest_silent_streak"] = longest_streak
    participants[name]["silent_start"] = longest_start
    participants[name]["silent_end"] = longest_end


# ================================================================
# 10. ACTIVITY SUMMARY
# ================================================================

total_participant_messages = sum(
    participants[name]["messages"] for name in participants
)

total_text_messages = sum(
    participants[name]["text_messages"] for name in participants
)

total_media = sum(
    participants[name]["media"] for name in participants
)

total_deleted = sum(
    participants[name]["deleted"] for name in participants
)

total_active_days = len(
    set(
        item["timestamp"].date()
        for item in all_messages
    )
)

total_characters = sum(
    participants[name]["total_characters"]
    for name in participants
)

average_length = safe_divide(total_characters, total_text_messages)

leaderboard = sorted(
    participants.items(),
    key=lambda entry: entry[1]["messages"],
    reverse=True
)

hour_totals = [0] * 24
weekday_totals = [0] * 7

for name in participants:
    for hour in range(24):
        hour_totals[hour] += participants[name]["hour_counts"][hour]

    for weekday in range(7):
        weekday_totals[weekday] += participants[name]["day_counts"][weekday]

most_active_hour = hour_totals.index(max(hour_totals))
most_active_weekday = weekday_totals.index(max(weekday_totals))

WEEKDAYS = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]


# ================================================================
# 11. NUMPY ACTIVITY HEATMAP
# ================================================================

# The NumPy matrix is real numerical data, not a Python list
# pretending to be a matrix.
#
# Rows = participants
# Columns = hours 00 through 23

known_participants = [
    name for name in EXPECTED_PARTICIPANTS
    if name in participants
]

activity_matrix = np.zeros(
    (len(known_participants), 24),
    dtype=int
)

for row, name in enumerate(known_participants):
    activity_matrix[row, :] = np.array(
        participants[name]["hour_counts"],
        dtype=int
    )

row_totals = np.sum(activity_matrix, axis=1)
column_totals = np.sum(activity_matrix, axis=0)

max_cell = int(np.max(activity_matrix)) if activity_matrix.size else 0

# Visualize the matrix with text characters.
# No Matplotlib or Seaborn is used.

HEAT_SYMBOLS = ["·", "░", "▒", "▓", "█"]

def heat_symbol(value, maximum):
    if maximum == 0:
        return HEAT_SYMBOLS[0]

    ratio = value / maximum

    if ratio == 0:
        return HEAT_SYMBOLS[0]
    elif ratio < 0.20:
        return HEAT_SYMBOLS[1]
    elif ratio < 0.40:
        return HEAT_SYMBOLS[2]
    elif ratio < 0.70:
        return HEAT_SYMBOLS[3]
    else:
        return HEAT_SYMBOLS[4]


# ================================================================
# 12. PERSONALITY ARCHETYPES
# ================================================================

def get_archetype(name, person, maximum_messages):

    messages = person["messages"]
    active_days = len(person["active_days"])

    night_percentage = percentage(
        person["night_messages"],
        person["text_messages"]
    )

    drama_percentage = percentage(
        person["drama_messages"],
        person["text_messages"]
    )

    average_message_length = safe_divide(
        person["total_characters"],
        person["text_messages"]
    )

    # Dataset-oriented rules:
    # These rules are based on measurable behavior.
    # Thresholds can be adjusted after inspecting the real results.

    # 1. Ghost: consistently low participation.
    if active_days <= 18 and messages < maximum_messages * 0.10:
        return "👻 Ghost"

    # 2. Night Owl: unusually high night-time activity.
    if person["text_messages"] > 0 and night_percentage >= 60:
        return "🦉 Night Owl"

    # 3. Spammer: sends a very high share of the group's messages.
    if messages >= maximum_messages * 0.90:
        return "📨 Spammer"

    # 4. Drama Queen: high share of uppercase/exclamation messages.
    if person["text_messages"] > 0 and drama_percentage >= 8:
        return "🎭 Drama Queen"

    # 5. Storyteller: longer-than-average messages.
    if (
        person["text_messages"] >= 10
        and average_message_length >= average_length * 1.45
    ):
        return "📚 Storyteller"

    # 6. Question Master: asks questions frequently.
    question_percentage = percentage(
        person["questions"],
        person["text_messages"]
    )

    if person["text_messages"] > 0 and question_percentage >= 18:
        return "❓ Question Master"

    # 7. Group Mom: active, steady participant.
    if (
        active_days >= total_days * 0.55
        and messages >= maximum_messages * 0.45
    ):
        return "💚 Group Mom"

    # 8. Comedian: frequent laughter-related terms.
    laughter_count = 0

    for word in ["haha", "hahaha", "lol", "lmao"]:
        laughter_count += person["words"].get(word, 0)

    if laughter_count >= 10:
        return "😂 Comedian"

    # 9. Default category.
    return "💬 Regular Contributor"


maximum_messages = max(
    (person["messages"] for person in participants.values()),
    default=0
)

for name, person in participants.items():
    person["archetype"] = get_archetype(
        name,
        person,
        maximum_messages
    )


# ================================================================
# 13. GROUPDNA REPORT HEADER
# ================================================================

print("\n")
print("╔" + "═" * 74 + "╗")
print("║" + " " * 17 + "🧬 GROUPDNA ANALYTICS REPORT" + " " * 29 + "║")
print("║" + " " * 13 + "WHATSAPP GROUP PERSONALITY INTELLIGENCE" + " " * 22 + "║")
print("╚" + "═" * 74 + "╝")

print(f"\n  📂 File            : {os.path.basename(FILE_PATH)}")
print(f"  🗓️  Date range      : {first_date} to {last_date}")
print(f"  📅 Days analyzed   : {total_days}")
print(f"  👥 Participants    : {len(participants)}")
print(f"  💬 Total messages  : {total_participant_messages:,}")
print(f"  📝 Text messages   : {total_text_messages:,}")
print(f"  🖼️  Media entries   : {total_media:,}")
print(f"  🗑️  Deleted entries : {total_deleted:,}")
print(f"  ⚙️  System messages : {system_messages:,}")


# ================================================================
# 14. SECTION A — GROUP OVERVIEW
# ================================================================

heading("01. GROUP OVERVIEW", "📊")

print(f"  Raw file lines          : {raw_line_count:,}")
print(f"  Participant messages    : {total_participant_messages:,}")
print(f"  Ordinary text messages  : {total_text_messages:,}")
print(f"  Media-omitted messages  : {total_media:,}")
print(f"  Deleted messages        : {total_deleted:,}")
print(f"  System messages         : {system_messages:,}")
print(f"  Empty lines skipped     : {empty_lines:,}")
print(f"  Continuation lines      : {continuation_lines:,}")
print(f"  Malformed records       : {malformed_lines:,}")
print(f"  Average text length     : {average_length:.1f} characters")
print(f"  Days with activity      : {total_active_days}/{total_days}")

if total_participant_messages:
    print(
        f"  Media share             : "
        f"{percentage(total_media, total_participant_messages):.2f}%"
    )

    print(
        f"  Deleted share           : "
        f"{percentage(total_deleted, total_participant_messages):.2f}%"
    )


# ================================================================
# 15. SECTION B — PARTICIPANT LEADERBOARD
# ================================================================

heading("02. PARTICIPANT LEADERBOARD", "🏆")

for rank, (name, person) in enumerate(leaderboard, start=1):

    count = person["messages"]
    share = percentage(count, total_participant_messages)

    print(
        f"\n  {rank:>2}. {name:<10} "
        f"{count:>5,} messages  "
        f"{share:>6.2f}%"
    )

    print(f"      {bar(count, maximum_messages, 34)}")

    print(
        f"      Text: {person['text_messages']:,} | "
        f"Media: {person['media']} | "
        f"Deleted: {person['deleted']} | "
        f"Active days: {len(person['active_days'])}"
    )


# ================================================================
# 16. SECTION C — MEDIA AND DELETED MESSAGE ANALYSIS
# ================================================================

heading("03. MEDIA & DELETED MESSAGE ANALYSIS", "🗂️")

section("Media sharing")

media_leaderboard = sorted(
    participants.items(),
    key=lambda entry: entry[1]["media"],
    reverse=True
)

for name, person in media_leaderboard:
    print(
        f"  {name:<10} {person['media']:>4} media entries  "
        f"{bar(person['media'], max(1, total_media), 20)}"
    )

section("Deleted messages")

deleted_leaderboard = sorted(
    participants.items(),
    key=lambda entry: entry[1]["deleted"],
    reverse=True
)

for name, person in deleted_leaderboard:
    print(
        f"  {name:<10} {person['deleted']:>4} deleted entries  "
        f"{bar(person['deleted'], max(1, total_deleted), 20)}"
    )


# ================================================================
# 17. SECTION D — WORD FREQUENCY
# ================================================================

heading("04. WHAT DOES THE GROUP TALK ABOUT?", "🔤")

group_words = {}

for name in participants:
    for word, count in participants[name]["words"].items():
        group_words[word] = group_words.get(word, 0) + count

top_words = sorted(
    group_words.items(),
    key=lambda entry: (-entry[1], entry[0])
)[:15]

if top_words:
    maximum_word_count = top_words[0][1]

    for rank, (word, count) in enumerate(top_words, start=1):
        print(
            f"  {rank:>2}. {word:<18} {count:>5,}  "
            f"{bar(count, maximum_word_count, 25)}"
        )
else:
    print("  No words available after tokenization and stop-word filtering.")


# ================================================================
# 18. SECTION E — MOST ACTIVE DAY AND HOUR
# ================================================================

heading("05. WHEN IS THE GROUP MOST ACTIVE?", "⏰")

print(
    f"\n  🔥 Peak hour: {most_active_hour:02d}:00–"
    f"{(most_active_hour + 1) % 24:02d}:00"
)
print(f"     Messages: {hour_totals[most_active_hour]:,}")

print(f"\n  📆 Most active weekday: {WEEKDAYS[most_active_weekday]}")
print(f"     Messages: {weekday_totals[most_active_weekday]:,}")

section("Hourly activity")

for hour in range(24):
    print(
        f"  {hour:02d}:00  "
        f"{bar(hour_totals[hour], max(hour_totals), 28)} "
        f"{hour_totals[hour]:>5,}"
    )


# ================================================================
# 19. SECTION F — NUMPY ACTIVITY HEATMAP
# ================================================================

heading("06. NUMPY ACTIVITY HEATMAP", "🌡️")

print("\n  Rows = participants | Columns = hours of the day")
print("  · none  ░ low  ▒ moderate  ▓ high  █ very high\n")

print("  Participant  " + " ".join(f"{hour:02d}" for hour in range(24)))
line()

for row, name in enumerate(known_participants):

    symbols = [
        heat_symbol(int(value), max_cell)
        for value in activity_matrix[row]
    ]

    row_total = int(row_totals[row])

    print(
        f"  {name:<12} "
        + "  ".join(symbols)
        + f"  | {row_total:>5,}"
    )

line()

print("  Hour total   " + " ".join(
    f"{int(value):02d}" if value < 100 else "**"
    for value in column_totals
))

print("\n  NumPy matrix shape:", activity_matrix.shape)
print("  NumPy matrix total:", int(np.sum(activity_matrix)))

print("\n  Row-total validation:")

for row, name in enumerate(known_participants):
    expected = participants[name]["messages"]
    actual = int(row_totals[row])

    # Media and deleted entries are included in the matrix, so
    # row totals should equal all participant messages.
    result = "PASS" if actual == expected else "CHECK"

    print(
        f"  {name:<10} matrix={actual:>5,} | "
        f"messages={expected:>5,} | {result}"
    )


# ================================================================
# 20. SECTION G — NIGHT OWL ANALYSIS
# ================================================================

heading("07. NIGHT-OWL DETECTION", "🌙")

night_leaderboard = sorted(
    participants.items(),
    key=lambda entry: percentage(
        entry[1]["night_messages"],
        entry[1]["text_messages"]
    ),
    reverse=True
)

for name, person in night_leaderboard:

    night_pct = percentage(
        person["night_messages"],
        person["text_messages"]
    )

    print(
        f"  {name:<10} {night_pct:>6.2f}% night messages  "
        f"{bar(night_pct, 100, 24)}"
    )

    print(
        f"             {person['night_messages']} messages "
        f"between 23:00 and 03:59"
    )


# ================================================================
# 21. SECTION H — RESPONSE TIME
# ================================================================

heading("08. RESPONSE-TIME ANALYSIS", "⚡")

print(
    "\n  Note: these are gaps between consecutive messages from "
    "different senders."
)
print("  They are not guaranteed to be direct replies.\n")

response_summary = []

for name in participants:

    gaps = response_gaps[name]

    if gaps:
        average_gap = sum(gaps) / len(gaps)
        sorted_gaps = sorted(gaps)
        median_gap = sorted_gaps[len(sorted_gaps) // 2]

        response_summary.append(
            (name, average_gap, median_gap, len(gaps))
        )

    else:
        average_gap = 0
        median_gap = 0

        response_summary.append(
            (name, average_gap, median_gap, 0)
        )

response_summary.sort(key=lambda entry: entry[1])

for name, avg_gap, median_gap, gap_count in response_summary:

    print(f"  👤 {name}")

    if gap_count == 0:
        print("     No measurable response gaps.")
    else:
        print(f"     Average gap : {avg_gap:.2f} minutes")
        print(f"     Median gap  : {median_gap:.2f} minutes")
        print(f"     Gaps counted: {gap_count:,}")


if all_response_gaps:
    group_average_gap = sum(all_response_gaps) / len(all_response_gaps)

    sorted_all_gaps = sorted(all_response_gaps)
    group_median_gap = sorted_all_gaps[len(sorted_all_gaps) // 2]

    print("\n  GROUP RESPONSE SUMMARY")
    print(f"  Average gap: {group_average_gap:.2f} minutes")
    print(f"  Median gap : {group_median_gap:.2f} minutes")
    print(f"  Gaps       : {len(all_response_gaps):,}")


# ================================================================
# 22. SECTION I — SILENT STREAKS
# ================================================================

heading("09. LONGEST SILENT STREAKS", "👻")

silent_ranking = sorted(
    participants.items(),
    key=lambda entry: entry[1]["longest_silent_streak"],
    reverse=True
)

for name, person in silent_ranking:

    streak = person["longest_silent_streak"]

    print(
        f"  {name:<10} {streak:>3} consecutive silent days  "
        f"{bar(streak, max(1, total_days), 25)}"
    )

    if streak > 0:
        print(
            f"             {person['silent_start']} to "
            f"{person['silent_end']}"
        )
    else:
        print("             No silent days in the analysis period.")


# ================================================================
# 23. SECTION J — PERSONALITY ARCHETYPES
# ================================================================

heading("10. GROUP PERSONALITY ARCHETYPES", "🧬")

for name, person in leaderboard:

    print(f"\n  👤 {name}")
    print(f"     Archetype: {person['archetype']}")
    print(f"     Messages : {person['messages']:,}")
    print(f"     Active   : {len(person['active_days'])} days")

    night_pct = percentage(
        person["night_messages"],
        person["text_messages"]
    )

    drama_pct = percentage(
        person["drama_messages"],
        person["text_messages"]
    )

    avg_length = safe_divide(
        person["total_characters"],
        person["text_messages"]
    )

    print(
        f"     Night activity: {night_pct:.1f}% | "
        f"Drama messages: {drama_pct:.1f}% | "
        f"Average text length: {avg_length:.1f}"
    )


# ================================================================
# 24. SECTION K — MESSAGE STYLE
# ================================================================

heading("11. MESSAGE STYLE INSIGHTS", "📝")

for name, person in leaderboard:

    avg_length = safe_divide(
        person["total_characters"],
        person["text_messages"]
    )

    uppercase_pct = percentage(
        person["uppercase_messages"],
        person["text_messages"]
    )

    drama_pct = percentage(
        person["drama_messages"],
        person["text_messages"]
    )

    question_pct = percentage(
        person["questions"],
        person["text_messages"]
    )

    print(f"\n  {name}")
    print(f"     Average message length : {avg_length:.1f} characters")
    print(f"     Uppercase messages     : {uppercase_pct:.1f}%")
    print(f"     Drama-rule messages    : {drama_pct:.1f}%")
    print(f"     Messages with '?'      : {question_pct:.1f}%")


# ================================================================
# 25. SECTION L — AUTOMATIC EXECUTIVE INSIGHTS
# ================================================================

heading("12. EXECUTIVE INSIGHTS", "💡")

if leaderboard:

    top_name, top_person = leaderboard[0]

    print(
        f"\n  🏆 Most active participant: {top_name}, "
        f"with {top_person['messages']:,} messages."
    )

    if len(leaderboard) > 1:
        least_name, least_person = leaderboard[-1]

        print(
            f"  👻 Least active participant: {least_name}, "
            f"with {least_person['messages']:,} messages."
        )

    night_name, night_person = night_leaderboard[0]

    night_pct = percentage(
        night_person["night_messages"],
        night_person["text_messages"]
    )

    print(
        f"  🌙 Highest night-activity share: {night_name} "
        f"({night_pct:.1f}%)."
    )

    if deleted_leaderboard:
        deleted_name, deleted_person = deleted_leaderboard[0]

        print(
            f"  🗑️ Most deleted messages: {deleted_name} "
            f"({deleted_person['deleted']})."
        )

    if media_leaderboard:
        media_name, media_person = media_leaderboard[0]

        print(
            f"  🖼️ Most media entries: {media_name} "
            f"({media_person['media']})."
        )

    if silent_ranking:
        silent_name, silent_person = silent_ranking[0]

        print(
            f"  💤 Longest silent streak: {silent_name} "
            f"({silent_person['longest_silent_streak']} days)."
        )

    print(
        f"  💬 Average ordinary text length: "
        f"{average_length:.1f} characters."
    )

    print(
        f"  ⏱️ Total measurable response gaps: "
        f"{len(all_response_gaps):,}."
    )


# ================================================================
# 26. DATA QUALITY CHECKS
# ================================================================

heading("13. DATA QUALITY & VALIDATION", "🔍")

print(f"\n  File lines              : {raw_line_count:,}")
print(f"  Participant messages   : {total_participant_messages:,}")
print(f"  System messages        : {system_messages:,}")
print(f"  Empty lines            : {empty_lines:,}")
print(f"  Continuation lines     : {continuation_lines:,}")
print(f"  Malformed records      : {malformed_lines:,}")
print(f"  Known expected names   : {len(EXPECTED_PARTICIPANTS)}")
print(f"  Names detected         : {len(participants)}")

missing_names = [
    name for name in EXPECTED_PARTICIPANTS
    if name not in participants
]

unexpected_names = [
    name for name in participants
    if name not in EXPECTED_PARTICIPANTS
]

if missing_names:
    print("\n  ⚠️ Expected participants not found:", ", ".join(missing_names))
else:
    print("\n  ✅ All expected participants were found.")

if unexpected_names:
    print(
        "  ⚠️ Additional sender names detected:",
        ", ".join(unexpected_names)
    )
else:
    print("  ✅ No unexpected sender names detected.")

if total_participant_messages == 3174:
    print("  ✅ Participant-message count matches the brief.")
else:
    print(
        f"  ⚠️ Participant-message count differs from the brief's "
        f"reference value of 3,174. Actual: "
        f"{total_participant_messages:,}."
    )

if system_messages == 4:
    print("  ✅ System-message count matches the brief.")
else:
    print(
        f"  ⚠️ System-message count differs from the reference "
        f"value of 4. Actual: {system_messages:,}."
    )

if activity_matrix.size:
    matrix_total = int(np.sum(activity_matrix))

    if matrix_total == sum(
        participants[name]["messages"]
        for name in known_participants
    ):
        print("  ✅ NumPy activity matrix total is consistent.")
    else:
        print("  ⚠️ Check the NumPy activity matrix totals.")


# ================================================================
# 27. FINAL REPORT FOOTER
# ================================================================

print("\n")
print("╔" + "═" * 74 + "╗")
print("║" + " " * 23 + "GROUPDNA REPORT COMPLETE" + " " * 27 + "║")
print("║" + " " * 18 + "DATA  •  PATTERNS  •  PERSONALITY" + " " * 23 + "║")
print("╚" + "═" * 74 + "╝")

print("\n  Analysis completed successfully.")
print("  Libraries used: NumPy and Python standard library.")
print("  pandas: NOT USED")
print("  Matplotlib: NOT USED")
print("  Seaborn: NOT USED")
print("  Regular expressions (re): NOT USED")
print("\n  Tip: capture screenshots of the dashboard for your project README.")


📁 Dataset loaded: hostel_bois.txt


╔══════════════════════════════════════════════════════════════════════════╗
║                 🧬 GROUPDNA ANALYTICS REPORT                             ║
║             WHATSAPP GROUP PERSONALITY INTELLIGENCE                      ║
╚══════════════════════════════════════════════════════════════════════════╝

  📂 File            : hostel_bois.txt
  🗓️  Date range      : 2024-04-01 to 2024-05-30
  📅 Days analyzed   : 60
  👥 Participants    : 6
  💬 Total messages  : 3,174
  📝 Text messages   : 3,127
  🖼️  Media entries   : 32
  🗑️  Deleted entries : 15
  ⚙️  System messages : 4

════════════════════════════════════════════════════════════════════════════
  📊 01. GROUP OVERVIEW
════════════════════════════════════════════════════════════════════════════
  Raw file lines          : 3,178
  Participant messages    : 3,174
  Ordinary text messages  : 3,127
  Media-omitted messages  : 32
  Deleted messages        : 15
  System messages         : 4
  Empty lin